# 📅 Rodada Atual v3 — Probabilidades + EV em Tempo Real
### Notebook 2 de 2: Jogos futuros · Prob base · Odds D-2 · Scheduler automático

**Fluxo de uso:**
```
MANHÃ (1x por dia)
  └── Rodar Células 1→6
  └── Gera rodada_atual_v3.csv com prob_base SEM odd
  └── Salva odds_d2 (referência para medir movimento)

30 MIN ANTES DE CADA JOGO (automático)
  └── Scheduler (Célula 7) roda em background
  └── Busca odd atual da API
  └── Calcula EV = prob_base × odd_atual - 1
  └── Calcula movimento = odd_d2 - odd_atual
  └── Atualiza rodada_atual_v3.csv
  └── Alerta no terminal quando EV > 3%
```

**Dependências:**
- `modelo_v3.pkl` — gerado pelo Notebook 1
- `elo_ratings.json` — gerado pelo Notebook 1


In [1]:
import requests, pandas as pd, numpy as np
import time, os, json, pickle, warnings
from datetime import datetime, timedelta
import pytz
warnings.filterwarnings("ignore")

# ── Configuração ──────────────────────────────────────────────────────────────
API_KEY    = "033f6d5fc7b707b144fbd2c28898c1540eb066ee9efea6c038d740ffedeacb5f"
BASE_URL   = "https://api.football-data-api.com"
PKL_FILE   = "modelo_v3.pkl"
ELO_FILE   = "elo_ratings.json"
RODADA_CSV = "rodada_atual_v3.csv"
JANELA     = 10
ELO_HOME   = 50
ELO_BASE   = 1500
EV_MIN     = 0.03        # 3% de edge mínimo
SLEEP      = 0.5
MAX_PAGE   = 300

# Fuso horário (ajuste para o seu)
TZ = pytz.timezone("America/Sao_Paulo")

# ── Ligas e season_ids ────────────────────────────────────────────────────────
LIGAS = {
    "Alemanha_1 Bundesliga":        14968,
    "Alemanha_2 2. Bundesliga":     14931,
    "Alemanha_3 3. Liga":           14977,
    "Australia_1 A-League":         16036,
    "Belgica_1 Pro League":         14937,
    "Bulgaria_1 First League":      15056,
    "Croacia_1 HNL":                15053,
    "Escocia_1 Premiership":        15000,
    "Espanha_1 La Liga":            14956,
    "Espanha_2 Segunda Division":   15066,
    "Franca_1 Ligue 1":             14932,
    "Franca_2 Ligue 2":             14954,
    "Grecia_1 Super League":        15163,
    "Holanda_1 Eredivisie":         14936,
    "Holanda_2 Eerste Divisie":     14987,
    "Hungria_1 NB I":               14963,
    "Inglaterra_1 Premier League":  15050,
    "Inglaterra_2 Championship":    14930,
    "Inglaterra_3 League One":      14934,
    "Italia_1 Serie A":             15068,
    "Italia_2 Serie B":             15632,
    "Portugal_1 Primeira Liga":     15115,
    "Rep_Tcheca_1 Czech Liga":      14973,
    "Turquia_1 Super Lig":          14972,
    "Brasileiro_A Serie A":         16544,
    "Argentina_1 Liga Profesional": 16571,
    "Chile_1 Primera Division":     16615,
    "Colombia_1 Liga BetPlay":      16614,
}

print(f"✅ Configuração carregada | {len(LIGAS)} ligas")
print(f"   EV mínimo: {EV_MIN*100:.0f}% | Janela: N{JANELA}")


✅ Configuração carregada | 28 ligas
   EV mínimo: 3% | Janela: N10


In [2]:
# ── Carregar modelo treinado e ratings ELO ───────────────────────────────────

# Modelo
if not os.path.exists(PKL_FILE):
    raise FileNotFoundError(f"❌ {PKL_FILE} não encontrado. Rode o Notebook 1 primeiro.")
with open(PKL_FILE, "rb") as f:
    modelos = pickle.load(f)
print(f"✅ Modelo carregado: {len(modelos)} mercados")
for label, m in modelos.items():
    print(f"   {m['nome']:<16} AUC={m['auc']:.3f}")

# ELO ratings
if not os.path.exists(ELO_FILE):
    raise FileNotFoundError(f"❌ {ELO_FILE} não encontrado. Rode o Notebook 1 primeiro.")
with open(ELO_FILE, "r") as f:
    elo_ratings = json.load(f)
print(f"\n✅ ELO carregado: {len(elo_ratings)} times")

def get_elo(league, team_id):
    return elo_ratings.get(f"{league}|{str(team_id)}", ELO_BASE)


✅ Modelo carregado: 6 mercados
   Back Casa        AUC=0.655
   Back Fora        AUC=0.649
   BTTS             AUC=0.513
   Over 2.5         AUC=0.514
   Over 0.5 HT      AUC=0.520
   BTTS HT          AUC=0.513

✅ ELO carregado: 509 times


In [3]:
# ── Buscar jogos futuros da API ───────────────────────────────────────────────
# Busca jogos com status != complete (agendados/ao vivo)
# Janela: hoje + próximos 7 dias

def buscar_jogos_futuros(liga_nome, season_id):
    jogos, page = [], 1
    agora  = int(datetime.now(TZ).timestamp())
    limite = agora + 7 * 86400  # 7 dias à frente

    while True:
        try:
            resp = requests.get(
                f"{BASE_URL}/league-matches",
                params={"key": API_KEY, "season_id": season_id,
                        "max_per_page": MAX_PAGE, "page": page},
                timeout=20
            )
            if resp.status_code == 429:
                print("  ⚠️  Rate limit — aguardando 60s...")
                time.sleep(60); continue
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  ❌ {liga_nome}: {e}"); break

        matches = data.get("data", [])
        if not matches: break

        for m in matches:
            status     = m.get("status", "")
            date_unix  = m.get("date_unix", 0) or 0

            # Só jogos futuros dentro da janela
            if status == "complete": continue
            if date_unix < agora or date_unix > limite: continue

            hid = m.get("homeID")
            aid = m.get("awayID")
            hid = str(int(hid)) if hid is not None else "0"
            aid = str(int(aid)) if aid is not None else "0"

            jogos.append({
                "fixture_id":    str(m.get("id", "")),
                "league":        liga_nome,
                "season_id":     season_id,
                "date_unix":     date_unix,
                "home_id":       hid,
                "away_id":       aid,
                "home_team":     m.get("home_name", ""),
                "away_team":     m.get("away_name", ""),
                "status":        status,
                # Odds D-2 (momento da coleta matinal)
                "odd_home_d2":   m.get("odds_ft_1"),
                "odd_draw_d2":   m.get("odds_ft_x"),
                "odd_away_d2":   m.get("odds_ft_2"),
                "odd_btts_d2":   m.get("odds_btts_yes"),
                "odd_over25_d2": m.get("odds_ft_over25"),
                # XG pré-jogo (disponível da API antes do jogo)
                "xg_home_pre":   m.get("team_a_xg_prematch"),
                "xg_away_pre":   m.get("team_b_xg_prematch"),
                "ppg_home":      m.get("pre_match_home_ppg"),
                "ppg_away":      m.get("pre_match_away_ppg"),
            })

        pager = data.get("pager", {})
        if page >= pager.get("total_pages", 1): break
        page += 1
        time.sleep(SLEEP)
    return jogos

# Executar busca
print("🔍 Buscando jogos futuros (próximos 7 dias)...")
todos = []
for liga_nome, season_id in LIGAS.items():
    jogos = buscar_jogos_futuros(liga_nome, season_id)
    if jogos:
        print(f"  📅 {liga_nome}: {len(jogos)} jogos")
    todos.extend(jogos)
    time.sleep(SLEEP)

df_jogos = pd.DataFrame(todos) if todos else pd.DataFrame()
print(f"\n✅ Total: {len(df_jogos)} jogos nos próximos 7 dias")
if len(df_jogos) > 0:
    df_jogos["datahora"] = pd.to_datetime(df_jogos["date_unix"], unit="s")                               .dt.tz_localize("UTC").dt.tz_convert(TZ)                               .dt.strftime("%Y-%m-%d %H:%M")
    display(df_jogos[["datahora","league","home_team","away_team",
                       "odd_home_d2","odd_away_d2","xg_home_pre","xg_away_pre"]].head(10))


🔍 Buscando jogos futuros (próximos 7 dias)...
  📅 Alemanha_1 Bundesliga: 9 jogos
  📅 Alemanha_2 2. Bundesliga: 9 jogos
  📅 Alemanha_3 3. Liga: 8 jogos
  📅 Australia_1 A-League: 5 jogos
  📅 Belgica_1 Pro League: 8 jogos
  📅 Bulgaria_1 First League: 7 jogos
  📅 Croacia_1 HNL: 5 jogos
  📅 Escocia_1 Premiership: 6 jogos
  📅 Espanha_1 La Liga: 7 jogos
  📅 Espanha_2 Segunda Division: 9 jogos
  📅 Franca_1 Ligue 1: 9 jogos
  📅 Franca_2 Ligue 2: 9 jogos
  📅 Grecia_1 Super League: 7 jogos
  📅 Holanda_1 Eredivisie: 9 jogos
  📅 Holanda_2 Eerste Divisie: 10 jogos
  📅 Hungria_1 NB I: 6 jogos
  📅 Inglaterra_1 Premier League: 6 jogos
  📅 Inglaterra_2 Championship: 1 jogos
  📅 Inglaterra_3 League One: 2 jogos
  📅 Italia_1 Serie A: 7 jogos
  📅 Italia_2 Serie B: 2 jogos
  📅 Portugal_1 Primeira Liga: 9 jogos
  📅 Brasileiro_A Serie A: 9 jogos
  📅 Argentina_1 Liga Profesional: 14 jogos
  📅 Colombia_1 Liga BetPlay: 11 jogos

✅ Total: 184 jogos nos próximos 7 dias


,datahora,league,home_team,away_team,odd_home_d2,odd_away_d2,xg_home_pre,xg_away_pre
0,2026-03-21 11:30,Alemanha_1 Bundesliga,Bayern München,Union Berlin,1.08,13.00,2.43,1.34
1,2026-03-22 11:30,Alemanha_1 Bundesliga,Mainz 05,Eintracht Frankfurt,2.10,3.20,1.56,1.24
2,2026-03-22 15:30,Alemanha_1 Bundesliga,Augsburg,Stuttgart,3.60,1.91,1.41,1.79
3,2026-03-22 13:30,Alemanha_1 Bundesliga,St. Pauli,Freiburg,2.70,2.62,1.33,1.17
4,2026-03-21 14:30,Alemanha_1 Bundesliga,Borussia Dortmund,Hamburger SV,1.36,7.00,1.69,1.28
5,2026-03-20 16:30,Alemanha_1 Bundesliga,RB Leipzig,Hoffenheim,1.89,3.50,1.96,1.42
6,2026-03-21 11:30,Alemanha_1 Bundesliga,Heidenheim,Bayer Leverkusen,5.00,1.57,1.28,1.40
7,2026-03-21 11:30,Alemanha_1 Bundesliga,Köln,Borussia M'gladbach,2.30,3.07,1.53,1.25
8,2026-03-21 11:30,Alemanha_1 Bundesliga,Wolfsburg,Werder Bremen,2.37,2.88,1.54,1.30
9,2026-03-20 14:30,Alemanha_2 2. Bundesliga,Hannover 96,Eintracht Braunschweig,1.55,5.00,1.95,1.36


In [4]:
# ── Buscar features N10 dos times a partir da base histórica ─────────────────
# Pega os últimos N10 jogos de cada time da base histórica
# e calcula as médias que o modelo precisa

RAW_CSV = "raw_2526.csv"

if not os.path.exists(RAW_CSV):
    raise FileNotFoundError(f"❌ {RAW_CSV} não encontrado. Rode o Notebook 1 primeiro.")

df_hist = pd.read_csv(RAW_CSV, dtype={"home_id": str, "away_id": str})
df_hist["date_unix"] = pd.to_numeric(df_hist["date_unix"], errors="coerce")
df_hist = df_hist[df_hist["score_home"].notna()]  # só jogos encerrados
df_hist = df_hist.sort_values("date_unix")

def ultimos_n_jogos(df_hist, team_id, liga, n=10):
    """
    Retorna os últimos N jogos do time (como home ou away) numa liga.
    Considera perspectiva do time: gols_marcados = gols como home OU away.
    """
    mask_h = (df_hist["home_id"] == str(team_id)) & (df_hist["league"] == liga)
    mask_a = (df_hist["away_id"] == str(team_id)) & (df_hist["league"] == liga)

    jogos_h = df_hist[mask_h].copy()
    jogos_h["gols_marc"] = jogos_h["score_home"]
    jogos_h["gols_sof"]  = jogos_h["score_away"]
    jogos_h["gols_ht"]   = jogos_h["score_home_ht"].fillna(0)
    jogos_h["vitoria"]   = (jogos_h["score_home"] > jogos_h["score_away"]).astype(int)
    jogos_h["xg"]        = pd.to_numeric(jogos_h.get("xg_home", 0), errors="coerce").fillna(0)

    jogos_a = df_hist[mask_a].copy()
    jogos_a["gols_marc"] = jogos_a["score_away"]
    jogos_a["gols_sof"]  = jogos_a["score_home"]
    jogos_a["gols_ht"]   = jogos_a["score_away_ht"].fillna(0)
    jogos_a["vitoria"]   = (jogos_a["score_away"] > jogos_a["score_home"]).astype(int)
    jogos_a["xg"]        = pd.to_numeric(jogos_a.get("xg_away", 0), errors="coerce").fillna(0)

    todos = pd.concat([jogos_h, jogos_a]).sort_values("date_unix").tail(n)
    if len(todos) == 0:
        return {}

    total_gols = todos["score_home"] + todos["score_away"]
    return {
        "n_jogos":       len(todos),
        "avg_xg":        todos["xg"].mean(),
        "pct_vitoria":   todos["vitoria"].mean(),
        "gols_marc":     todos["gols_marc"].mean(),
        "gols_sof":      todos["gols_sof"].mean(),
        "gols_ht":       todos["gols_ht"].mean(),
        "pct_btts":      ((todos["score_home"]>0)&(todos["score_away"]>0)).mean(),
        "pct_over25":    (total_gols > 2.5).mean(),
        "pct_over05ht":  ((todos["score_home_ht"].fillna(0)+todos["score_away_ht"].fillna(0))>0.5).mean(),
    }

# Calcular features N10 para cada time dos jogos futuros
print(f"⚙️  Calculando features N{JANELA} para {len(df_jogos)} jogos...")

features_jogos = []
for _, jogo in df_jogos.iterrows():
    fh = ultimos_n_jogos(df_hist, jogo["home_id"], jogo["league"], JANELA)
    fa = ultimos_n_jogos(df_hist, jogo["away_id"], jogo["league"], JANELA)

    # ELO pré-jogo
    elo_h = get_elo(jogo["league"], jogo["home_id"])
    elo_a = get_elo(jogo["league"], jogo["away_id"])

    # PPG
    ppg_h = float(jogo.get("ppg_home") or 0)
    ppg_a = float(jogo.get("ppg_away") or 0)

    # XG pré-jogo
    xg_pre_h = float(jogo.get("xg_home_pre") or 0)
    xg_pre_a = float(jogo.get("xg_away_pre") or 0)

    features_jogos.append({
        "fixture_id":       jogo["fixture_id"],
        # Features do modelo
        "elo_delta":        elo_h + ELO_HOME - elo_a,
        "xg_n10_diff":      fh.get("avg_xg",0)      - fa.get("avg_xg",0),
        "xg_pre_diff":      xg_pre_h                 - xg_pre_a,
        "pct_vit_diff":     fh.get("pct_vitoria",0)  - fa.get("pct_vitoria",0),
        "ppg_diff":         ppg_h                    - ppg_a,
        "gols_marc_diff":   fh.get("gols_marc",0)    - fa.get("gols_marc",0),
        "gols_sof_diff":    fh.get("gols_sof",0)     - fa.get("gols_sof",0),
        # Dados extras para o painel
        "n10_home":         fh.get("n_jogos",0),
        "n10_away":         fa.get("n_jogos",0),
        "elo_home":         elo_h,
        "elo_away":         elo_a,
        "xg_home_n10":      fh.get("avg_xg",0),
        "xg_away_n10":      fa.get("avg_xg",0),
        "pct_vit_home":     fh.get("pct_vitoria",0),
        "pct_vit_away":     fa.get("pct_vitoria",0),
        "gols_marc_home":   fh.get("gols_marc",0),
        "gols_marc_away":   fa.get("gols_marc",0),
        "pct_over25_home":  fh.get("pct_over25",0),
        "pct_over25_away":  fa.get("pct_over25",0),
    })

df_feat_fut = pd.DataFrame(features_jogos)
df_jogos = df_jogos.merge(df_feat_fut, on="fixture_id", how="left")
print(f"✅ Features calculadas")
print(f"   Times sem histórico N{JANELA}: "
      f"{(df_jogos['n10_home']==0).sum() + (df_jogos['n10_away']==0).sum()} times")
display(df_jogos[["home_team","away_team","elo_delta","xg_n10_diff",
                   "xg_pre_diff","pct_vit_diff"]].head(10).round(3))


⚙️  Calculando features N10 para 184 jogos...
✅ Features calculadas
   Times sem histórico N10: 0 times


,home_team,away_team,elo_delta,xg_n10_diff,xg_pre_diff,pct_vit_diff
0,Bayern München,Union Berlin,268.295,0.886,1.09,0.5
1,Mainz 05,Eintracht Frankfurt,27.796,0.335,0.32,0.2
2,Augsburg,Stuttgart,-64.778,-0.328,-0.38,-0.1
3,St. Pauli,Freiburg,-6.223,-0.106,0.16,0.0
4,Borussia Dortmund,Hamburger SV,215.601,0.446,0.41,0.5
5,RB Leipzig,Hoffenheim,22.921,-0.077,0.54,-0.2
6,Heidenheim,Bayer Leverkusen,-160.583,-0.380,-0.12,-0.4
7,Köln,Borussia M'gladbach,8.319,0.174,0.28,0.0
8,Wolfsburg,Werder Bremen,27.507,-0.497,0.24,-0.1
9,Hannover 96,Eintracht Braunschweig,153.996,0.186,0.59,0.2


In [5]:
# ── Calcular probabilidades BASE (sem odd) ───────────────────────────────────
#
# A prob_base é calculada ANTES de olhar as odds.
# Isso garante que o modelo não fica viciado em odds atuais
# que podem mudar até o kickoff.
#
# A odd só entra no EV final (Célula 7 — scheduler).

FEATURES = ["elo_delta","xg_n10_diff","xg_pre_diff","pct_vit_diff",
            "ppg_diff","gols_marc_diff","gols_sof_diff"]

# Verificar quais jogos têm features completas
mask_ok = df_jogos[FEATURES].notna().all(axis=1)
print(f"Jogos com features completas: {mask_ok.sum()}/{len(df_jogos)}")

X = df_jogos.loc[mask_ok, FEATURES].values

for label, m_info in modelos.items():
    col_prob = m_info["col_prob"]
    scaler   = m_info["scaler"]
    model    = m_info["model"]

    X_sc  = scaler.transform(X)
    probs = model.predict_proba(X_sc)[:,1]
    df_jogos.loc[mask_ok, col_prob] = probs

# Calcular EV pré-jogo com odds D-2 (referência)
# EV definitivo virá do scheduler com odds do kickoff
for label, nome, odd_tipica, col_prob, col_ev in [
    ("label_home_win",  "Back Casa",   1.90, "prob_bc",      "ev_bc_d2"),
    ("label_away_win",  "Back Fora",   2.50, "prob_bf",      "ev_bf_d2"),
    ("label_btts",      "BTTS",        1.85, "prob_btts",    "ev_btts_d2"),
    ("label_over25",    "Over 2.5",    1.80, "prob_over25",  "ev_over25_d2"),
    ("label_over05ht",  "Over 0.5 HT", 1.40, "prob_o05ht",   "ev_o05ht_d2"),
    ("label_btts_ht",   "BTTS HT",     3.50, "prob_btts_ht", "ev_btts_ht_d2"),
]:
    odd_col = {"Back Casa":"odd_home_d2","Back Fora":"odd_away_d2",
               "BTTS":"odd_btts_d2","Over 2.5":"odd_over25_d2",
               "Over 0.5 HT":"odd_over25_d2","BTTS HT":"odd_btts_d2"}.get(nome)
    if odd_col and odd_col in df_jogos.columns:
        odd_vals = pd.to_numeric(df_jogos[odd_col], errors="coerce").fillna(odd_tipica)
        df_jogos[col_ev] = df_jogos[col_prob] * odd_vals - 1
    else:
        df_jogos[col_ev] = df_jogos[col_prob] * odd_tipica - 1

print("\n✅ Probabilidades base calculadas")
print("\n📊 Amostra — jogos com maior EV (Back Casa D-2):")
cols_show = ["datahora","league","home_team","away_team",
             "prob_bc","ev_bc_d2","elo_delta","xg_pre_diff"]
cols_show = [c for c in cols_show if c in df_jogos.columns]
top = df_jogos[cols_show].sort_values("ev_bc_d2", ascending=False).head(10)
display(top.round(3))


Jogos com features completas: 184/184

✅ Probabilidades base calculadas

📊 Amostra — jogos com maior EV (Back Casa D-2):


,datahora,league,home_team,away_team,prob_bc,ev_bc_d2,elo_delta,xg_pre_diff
82,2026-03-20 16:00,Franca_2 Ligue 2,Rodez,Bastia,0.641,0.602,208.478,0.35
95,2026-03-22 14:00,Grecia_1 Super League,Volos NFC,PAOK,0.145,0.594,-169.319,-0.14
15,2026-03-22 09:30,Alemanha_2 2. Bundesliga,Bochum,Holstein Kiel,0.649,0.298,155.663,0.63
151,2026-03-22 20:30,Brasileiro_A Serie A,Corinthians,Flamengo,0.373,0.269,38.814,-0.03
57,2026-03-21 12:15,Espanha_1 La Liga,RCD Espanyol,Getafe CF,0.517,0.261,95.049,0.44
158,2026-03-21 21:00,Brasileiro_A Serie A,São Paulo,Palmeiras,0.454,0.248,65.813,0.38
50,2026-03-22 11:00,Croacia_1 HNL,Slaven Koprivnica,Osijek,0.540,0.214,116.684,0.34
26,2026-03-22 05:00,Australia_1 A-League,Perth Glory FC,Melbourne City FC,0.360,0.206,8.935,0.17
56,2026-03-22 09:30,Escocia_1 Premiership,Dundee United,Celtic,0.215,0.206,-95.472,-0.02
119,2026-03-21 12:30,Hungria_1 NB I,Zalaegerszegi TE,Újpest,0.509,0.206,110.153,0.15


In [6]:
# ── Salvar rodada_atual_v3.csv ────────────────────────────────────────────────
# Este CSV é carregado pelo painel HTML e atualizado pelo scheduler

df_jogos.to_csv(RODADA_CSV, index=False)
print(f"💾 {RODADA_CSV} salvo!")
print(f"   {len(df_jogos)} jogos | {df_jogos['league'].nunique()} ligas")
print(f"   Prob cols: {[c for c in df_jogos.columns if c.startswith('prob_')]}")
print(f"   EV cols  : {[c for c in df_jogos.columns if c.startswith('ev_')]}")
print()

# Resumo de alertas D-2
print("🚨 ALERTAS D-2 — Jogos com EV > {:.0f}% (usando odds de hoje):".format(EV_MIN*100))
for col_ev, nome in [("ev_bc_d2","Back Casa"),("ev_bf_d2","Back Fora"),
                      ("ev_btts_d2","BTTS"),("ev_over25_d2","Over 2.5")]:
    if col_ev not in df_jogos.columns: continue
    alertas = df_jogos[df_jogos[col_ev] > EV_MIN].sort_values(col_ev, ascending=False)
    if len(alertas) > 0:
        print(f"\n  {nome} ({len(alertas)} jogos):")
        for _, r in alertas.head(5).iterrows():
            print(f"    {r.get('datahora','?')[:16]} | {r['home_team']} vs {r['away_team']}"
                  f" | EV={r[col_ev]*100:+.1f}%"
                  f" | prob={r[col_ev.replace('ev_','prob_').replace('_d2','')]*100:.1f}%"
                  if col_ev.replace('ev_','prob_').replace('_d2','') in r else
                  f"    {r.get('datahora','?')[:16]} | {r['home_team']} vs {r['away_team']}"
                  f" | EV={r[col_ev]*100:+.1f}%")

print("\n✅ Célula 6 concluída — pronto para o scheduler (Célula 7)")
print("   Dica: rodar Célula 7 nos dias de jogo para atualização automática")


💾 rodada_atual_v3.csv salvo!
   184 jogos | 25 ligas
   Prob cols: ['prob_bc', 'prob_bf', 'prob_btts', 'prob_over25', 'prob_o05ht', 'prob_btts_ht']
   EV cols  : ['ev_bc_d2', 'ev_bf_d2', 'ev_btts_d2', 'ev_over25_d2', 'ev_o05ht_d2', 'ev_btts_ht_d2']

🚨 ALERTAS D-2 — Jogos com EV > 3% (usando odds de hoje):

  Back Casa (45 jogos):
    2026-03-20 16:00 | Rodez vs Bastia | EV=+60.2% | prob=64.1%
    2026-03-22 14:00 | Volos NFC vs PAOK | EV=+59.4% | prob=14.5%
    2026-03-22 09:30 | Bochum vs Holstein Kiel | EV=+29.8% | prob=64.9%
    2026-03-22 20:30 | Corinthians vs Flamengo | EV=+26.9% | prob=37.3%
    2026-03-21 12:15 | RCD Espanyol vs Getafe CF | EV=+26.1% | prob=51.7%

  Back Fora (52 jogos):
    2026-03-22 14:00 | AEK Athens vs Kifisia | EV=+83.4% | prob=11.1%
    2026-03-23 21:15 | Huracán vs Barracas Central | EV=+69.8% | prob=33.9%
    2026-03-21 15:00 | Santa Clara vs Gil Vicente | EV=+50.1% | prob=50.7%
    2026-03-22 20:00 | Boca Juniors vs Instituto | EV=+48.0% | prob=29.6%


In [ ]:
# ── Scheduler — atualiza odds e EV a cada 30 minutos ─────────────────────────
#
# Como usar:
#   1. Rode esta célula nos dias de jogo
#   2. Ela roda em background no Jupyter
#   3. A cada 30 min busca odds atuais dos jogos que ainda não começaram
#   4. Recalcula EV com a odd do momento
#   5. Alerta no terminal quando EV > EV_MIN
#   6. Pressione o botão ■ (Stop) para parar
#
# Os arquivos rodada_atual_v3.csv são atualizados automaticamente
# O painel HTML lerá a versão mais recente

import threading

INTERVALO_MIN = 30  # minutos entre atualizações
_scheduler_ativo = True

def buscar_odd_atual(fixture_id, season_id):
    """Busca odd atual de um jogo específico via fixture_id."""
    try:
        resp = requests.get(
            f"{BASE_URL}/match",
            params={"key": API_KEY, "match_id": fixture_id},
            timeout=15
        )
        resp.raise_for_status()
        data = resp.json()
        m = data.get("data", {})
        if not m: return {}
        return {
            "odd_home_atual":   m.get("odds_ft_1"),
            "odd_draw_atual":   m.get("odds_ft_x"),
            "odd_away_atual":   m.get("odds_ft_2"),
            "odd_btts_atual":   m.get("odds_btts_yes"),
            "odd_over25_atual": m.get("odds_ft_over25"),
            "status":           m.get("status",""),
        }
    except:
        return {}

def calcular_ev_atual(prob, odd):
    """EV = prob × odd - 1. Retorna None se odd indisponível."""
    if not odd or odd <= 1.0: return None
    return prob * float(odd) - 1

def atualizar_ciclo():
    """Um ciclo completo de atualização de odds e EV."""
    agora_ts = int(datetime.now(TZ).timestamp())
    agora_str = datetime.now(TZ).strftime("%H:%M:%S")

    if not os.path.exists(RODADA_CSV):
        print(f"[{agora_str}] ❌ {RODADA_CSV} não encontrado")
        return

    df = pd.read_csv(RODADA_CSV, dtype={"fixture_id": str})
    # Só jogos que ainda não começaram (30 min de margem)
    pendentes = df[pd.to_numeric(df["date_unix"], errors="coerce") > agora_ts - 1800]

    if len(pendentes) == 0:
        print(f"[{agora_str}] ℹ️  Nenhum jogo pendente")
        return

    print(f"[{agora_str}] 🔄 Atualizando {len(pendentes)} jogos...")
    alertas = []

    for idx, jogo in pendentes.iterrows():
        fid     = str(jogo["fixture_id"])
        sid     = jogo.get("season_id", 0)
        odds_at = buscar_odd_atual(fid, sid)
        if not odds_at: continue
        time.sleep(0.3)

        # Atualizar status
        df.at[idx, "status"] = odds_at.get("status", df.at[idx, "status"])

        # Mapas de prob → odd atual → EV final
        mercado_map = [
            ("prob_bc",      "odd_home_atual",   "ev_bc",      "Back Casa",   "odd_home_d2"),
            ("prob_bf",      "odd_away_atual",   "ev_bf",      "Back Fora",   "odd_away_d2"),
            ("prob_btts",    "odd_btts_atual",   "ev_btts",    "BTTS",        "odd_btts_d2"),
            ("prob_over25",  "odd_over25_atual", "ev_over25",  "Over 2.5",    "odd_over25_d2"),
        ]

        for col_prob, col_odd_at, col_ev, nome, col_odd_d2 in mercado_map:
            if col_prob not in df.columns: continue
            prob     = float(df.at[idx, col_prob] or 0)
            odd_at   = odds_at.get(col_odd_at)
            odd_d2   = float(df.at[idx, col_odd_d2] or 0) if col_odd_d2 in df.columns else 0

            ev = calcular_ev_atual(prob, odd_at)
            if ev is None: continue

            df.at[idx, col_ev]              = ev
            df.at[idx, col_odd_at]          = odd_at

            # Movimento da odd (D-2 vs atual)
            if odd_d2 > 0 and odd_at:
                mov = ((float(odd_at) - odd_d2) / odd_d2) * 100
                df.at[idx, f"mov_{col_odd_at.replace('_atual','')}"] = mov

            # Alerta
            if ev > EV_MIN:
                time_ate = int((float(df.at[idx,"date_unix"]) - agora_ts) / 60)
                alertas.append({
                    "mercado": nome,
                    "jogo":    f"{jogo['home_team']} vs {jogo['away_team']}",
                    "liga":    jogo["league"],
                    "ev":      ev,
                    "prob":    prob,
                    "odd":     odd_at,
                    "min_ate": time_ate,
                    "mov":     ((float(odd_at)-odd_d2)/odd_d2*100) if odd_d2>0 and odd_at else 0,
                })

    df.to_csv(RODADA_CSV, index=False)

    # Exibir alertas
    if alertas:
        print(f"\n{'🚨'*3} ALERTAS DE VALOR {'🚨'*3}")
        alertas.sort(key=lambda x: x["ev"], reverse=True)
        for a in alertas:
            sinal_mov = "📉" if a["mov"] < -2 else "📈" if a["mov"] > 2 else "➡️"
            print(f"  ✅ {a['mercado']:<14} | {a['jogo']}")
            print(f"     Liga: {a['liga']} | Faltam: {a['min_ate']} min")
            print(f"     Prob={a['prob']*100:.1f}% | Odd={a['odd']} | EV={a['ev']*100:+.1f}%")
            print(f"     Movimento odd: {sinal_mov} {a['mov']:+.1f}%")
            print()
    else:
        print(f"[{agora_str}] ℹ️  Nenhum jogo com EV > {EV_MIN*100:.0f}% no momento")

    print(f"[{agora_str}] ✅ CSV atualizado → {RODADA_CSV}")

# ── Loop principal ────────────────────────────────────────────────────────────
print(f"🟢 Scheduler iniciado — atualizando a cada {INTERVALO_MIN} minutos")
print(f"   EV mínimo para alerta: {EV_MIN*100:.0f}%")
print(f"   Pressione ■ (Stop) para parar\n")

# Primeira execução imediata
atualizar_ciclo()

# Loop
ciclo = 0
while _scheduler_ativo:
    time.sleep(INTERVALO_MIN * 60)
    ciclo += 1
    print(f"\n── Ciclo {ciclo} ──────────────────────────────")
    atualizar_ciclo()


🟢 Scheduler iniciado — atualizando a cada 30 minutos
   EV mínimo para alerta: 3%
   Pressione ■ (Stop) para parar

[12:18:51] 🔄 Atualizando 184 jogos...


In [ ]:
# ── Parar o scheduler ────────────────────────────────────────────────────────
# Rode esta célula para parar o loop sem precisar reiniciar o kernel
_scheduler_ativo = False
print("🔴 Scheduler parado")
